# Fantasy Premier League points prediction

One pass from raw gameweek files to trained models, in the order it runs.

| stage | produces |
|---|---|
| 1. Load and merge | `all_seasons_data.csv` |
| 2. Match index | `game_number` |
| 3. Features | `all_seasons_data_featured.csv` |
| 4. Train | `saved_models/`, `model_metrics.json` |
| 5. Two-stage models | `P(plays)`, `E[points\|plays]`, `P(haul)` |
| 6. Save | artifacts for `scripts/predict_gameweek.py` |

The `scripts/` directory runs these same cells headlessly, one stage per
script, so this notebook stays the single definition of the pipeline.

Two environment switches:

- `FPL_FEATURE_SET=full` trains on all features instead of the compact set
- `FPL_SPLIT=xg` confines train/val/test to the seasons that record xG

## 1. Load and merge every season

Reads each season's merged_gw.csv, maps positions, teams and opponents to
names, harmonises columns that differ between seasons, and re-scores
2016-17 to 2018-19 under the current rules.

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

### Data Pipeline Notes

This section performs data transformations based on prior exploratory data analysis.

## Add Position Column

Map player element_type to position names (Goalkeeper, Defender, Midfielder, Forward).

In [ ]:
# Add position column to merged gameweek data based on element_type from player raw data
def add_position_to_merged_gw(merged_gw_df, cleaned_players_df):
    # Map element_type to position names
    element_type_to_position = {
        1: 'Goalkeeper',
        2: 'Defender',
        3: 'Midfielder',
        4: 'Forward'
    }
    # create a new column 'position' in cleaned players dataframe
    cleaned_players_df['position'] = cleaned_players_df['element_type'].map(element_type_to_position)
    # create a mapping from player id to position
    player_id_to_position = dict(zip(cleaned_players_df['id'], cleaned_players_df['position']))
    # add the position column to the merged gw dataframe
    merged_gw_df['position'] = merged_gw_df['element'].map(player_id_to_position)
    return merged_gw_df


### Load Player Raw Data

Load the players_raw.csv files containing player metadata for position mapping.

In [ ]:
# first load the raw player data for each season until 2019-20
data_players_2016_17 = pd.read_csv('data/2016-17/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2017_18 = pd.read_csv('data/2017-18/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2018_19 = pd.read_csv('data/2018-19/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2019_20 = pd.read_csv('data/2019-20/players_raw.csv', encoding='latin-1', on_bad_lines='skip')

## Apply Position Mapping

Add the position column to each season's gameweek data.

In [ ]:
# add the position column to each season's player data
data_merged_gw_2016_17 = add_position_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17)
data_merged_gw_2017_18 = add_position_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18)
data_merged_gw_2018_19 = add_position_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19)
data_merged_gw_2019_20 = add_position_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20)

## Add Team Names

Map team IDs to team names using the master team list for consistent team identification.

In [ ]:
# Add team names to seasons 2016-17 through 2019-20 using master team list
# Process:
# 1. Load master team list (contains season → team_id → team_name mapping)
# 2. Map player_id → team_id from player raw data
# 3. Map team_id → team_name from master list
# 4. Add team_name column to merged GW data
data_master_team_list = pd.read_csv('data/master_team_list.csv', encoding='latin-1', on_bad_lines='skip')
def add_team_name_to_merged_gw(merged_gw_df, players_raw_df, master_team_list_df, season):
    # filter the master team list for the given season
    season_team_list = master_team_list_df[master_team_list_df['season'] == season]
    # create a mapping from team id to team name
    team_id_to_name = dict(zip(season_team_list['team'], season_team_list['team_name']))
    # create a mapping from player id to team id
    player_id_to_team_id = dict(zip(players_raw_df['id'], players_raw_df['team']))
    # create a mapping from player id to team name
    player_id_to_team_name = {player_id: team_id_to_name.get(team_id, 'Unknown') for player_id, team_id in player_id_to_team_id.items()}

    # add the team name column to the merged gw dataframe    return merged_gw_df
    merged_gw_df['team'] = merged_gw_df['element'].map(player_id_to_team_name)

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

# Column Standardization

Ensure all seasons have consistent columns by removing season-specific attributes.

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

## Calculate Defensive Contribution

Compute defensive contribution scores based on FPL scoring rules per position.

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

In [ ]:
# Add placeholder columns (value 0) for defensive stats not tracked in seasons 2019-20 to 2024-25
# This ensures consistent schema across all seasons before calculating defensive_contribution
data_merged_gw_2019_20['clearances_blocks_interceptions'] = 0
data_merged_gw_2019_20['recoveries'] = 0
data_merged_gw_2019_20['tackles'] = 0
data_merged_gw_2020_21['clearances_blocks_interceptions'] = 0
data_merged_gw_2020_21['recoveries'] = 0
data_merged_gw_2020_21['tackles'] = 0
data_merged_gw_2021_22['clearances_blocks_interceptions'] = 0
data_merged_gw_2021_22['recoveries'] = 0
data_merged_gw_2021_22['tackles'] = 0
data_merged_gw_2022_23['clearances_blocks_interceptions'] = 0
data_merged_gw_2022_23['recoveries'] = 0
data_merged_gw_2022_23['tackles'] = 0
data_merged_gw_2023_24['clearances_blocks_interceptions'] = 0
data_merged_gw_2023_24['recoveries'] = 0
data_merged_gw_2023_24['tackles'] = 0
data_merged_gw_2024_25['clearances_blocks_interceptions'] = 0
data_merged_gw_2024_25['recoveries'] = 0
data_merged_gw_2024_25['tackles'] = 0
data_merged_gw_2016_17 = add_defensive_contribution(data_merged_gw_2016_17)
data_merged_gw_2017_18 = add_defensive_contribution(data_merged_gw_2017_18)
data_merged_gw_2018_19 = add_defensive_contribution(data_merged_gw_2018_19)
data_merged_gw_2019_20 = add_defensive_contribution(data_merged_gw_2019_20)
data_merged_gw_2020_21 = add_defensive_contribution(data_merged_gw_2020_21)
data_merged_gw_2021_22 = add_defensive_contribution(data_merged_gw_2021_22)
data_merged_gw_2022_23 = add_defensive_contribution(data_merged_gw_2022_23)
data_merged_gw_2023_24 = add_defensive_contribution(data_merged_gw_2023_24)
data_merged_gw_2024_25 = add_defensive_contribution(data_merged_gw_2024_25)

### Verify Column Consistency

Confirm all seasons have identical column sets after standardization.

In [ ]:
# compare the columns of all these datasets
merged_2016_17_columns = set(data_merged_gw_2016_17.columns.tolist())
merged_2017_18_columns = set(data_merged_gw_2017_18.columns.tolist())
merged_2018_19_columns = set(data_merged_gw_2018_19.columns.tolist())
merged_2019_20_columns = set(data_merged_gw_2019_20.columns.tolist())
merged_2020_21_columns = set(data_merged_gw_2020_21.columns.tolist())
merged_2021_22_columns = set(data_merged_gw_2021_22.columns.tolist())
merged_2022_23_columns = set(data_merged_gw_2022_23.columns.tolist())
merged_2023_24_columns = set(data_merged_gw_2023_24.columns.tolist())
merged_2024_25_columns = set(data_merged_gw_2024_25.columns.tolist())
merged_2025_26_columns = set(data_merged_gw_2025_26.columns.tolist())
# find the common columns across all seasons
common_merged_columns_all_seasons = merged_2016_17_columns.intersection(merged_2017_18_columns).intersection(merged_2018_19_columns).intersection(merged_2019_20_columns).intersection(merged_2020_21_columns).intersection(merged_2021_22_columns).intersection(merged_2022_23_columns).intersection(merged_2023_24_columns).intersection(merged_2024_25_columns).intersection(merged_2025_26_columns)
print("Common Columns Across All Seasons:", sorted(common_merged_columns_all_seasons))
# find the unique columns in each season compared to the common columns
unique_2016_17_columns = merged_2016_17_columns - common_merged_columns_all_seasons
unique_2017_18_columns = merged_2017_18_columns - common_merged_columns_all_seasons
unique_2018_19_columns = merged_2018_19_columns - common_merged_columns_all_seasons
unique_2019_20_columns = merged_2019_20_columns - common_merged_columns_all_seasons
unique_2020_21_columns = merged_2020_21_columns - common_merged_columns_all_seasons
unique_2021_22_columns = merged_2021_22_columns - common_merged_columns_all_seasons
unique_2022_23_columns = merged_2022_23_columns - common_merged_columns_all_seasons
unique_2023_24_columns = merged_2023_24_columns - common_merged_columns_all_seasons
unique_2024_25_columns = merged_2024_25_columns - common_merged_columns_all_seasons
unique_2025_26_columns = merged_2025_26_columns - common_merged_columns_all_seasons
print("Unique Columns in 2016-17:", sorted(unique_2016_17_columns))
print("Unique Columns in 2017-18:", sorted(unique_2017_18_columns))
print("Unique Columns in 2018-19:", sorted(unique_2018_19_columns))
print("Unique Columns in 2019-20:", sorted(unique_2019_20_columns))
print("Unique Columns in 2020-21:", sorted(unique_2020_21_columns))
print("Unique Columns in 2021-22:", sorted(unique_2021_22_columns))
print("Unique Columns in 2022-23:", sorted(unique_2022_23_columns))
print("Unique Columns in 2023-24:", sorted(unique_2023_24_columns))
print("Unique Columns in 2024-25:", sorted(unique_2024_25_columns))
print("Unique Columns in 2025-26:", sorted(unique_2025_26_columns))


# Historical Points Adjustment

Adjust historical season points to align with current FPL scoring rules.

## Key Design Decision

The points for previous seasons (2016-17 to 2018-19) are adjusted to match the current FPL scoring system that includes bonus points for defensive contributions.

### Implementation Note

For seasons 2019-20 to 2024-25, point modification will be applied after merging with defensive data.

### Point Modification Steps

1. Standardize position values across all seasons
2. Adjust points for seasons 2016-17 to 2018-19 to match current FPL defensive contribution scoring

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

### Deferred Point Modification

Points for seasons 2019-20 to 2025-26 will be modified after merging with defensive statistics data.

# Merge All Seasons Data

Combine all individual season datasets into a single unified DataFrame.

In [ ]:
# Add season identifier to each dataset before merging
data_merged_gw_2016_17['season'] = '2016-17'
data_merged_gw_2017_18['season'] = '2017-18'
data_merged_gw_2018_19['season'] = '2018-19'
data_merged_gw_2019_20['season'] = '2019-20'
data_merged_gw_2020_21['season'] = '2020-21'
data_merged_gw_2021_22['season'] = '2021-22'
data_merged_gw_2022_23['season'] = '2022-23'
data_merged_gw_2023_24['season'] = '2023-24'
data_merged_gw_2024_25['season'] = '2024-25'
data_merged_gw_2025_26['season'] = '2025-26'
# concatenating all seasons data into a single dataframe
all_seasons_data = pd.concat([data_merged_gw_2016_17, data_merged_gw_2017_18, data_merged_gw_2018_19, data_merged_gw_2019_20, data_merged_gw_2020_21, data_merged_gw_2021_22, data_merged_gw_2022_23, data_merged_gw_2023_24, data_merged_gw_2024_25, data_merged_gw_2025_26], ignore_index=True)
print("All Seasons Data Sample:")
print(all_seasons_data.sample(10))

## Standardize Position Codes

Convert position names to short codes (GK, DEF, MID, FWD) for consistency.

In [ ]:
# Standardize position values to short codes for consistency
all_seasons_data['position'] = all_seasons_data['position'].replace({'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD'})

## Convert Opponent Team IDs to Names

Map opponent team IDs to readable team names using season-specific mappings.

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
#
# data/master_team_list.csv only covers 2016-17..2023-24. For 2024-25 and
# 2025-26 the season lookup came back empty and `.get(team_id, team_id)` fell
# through to the raw numeric id, leaving opponent_team as "1", "2", ... for
# those two seasons.
#
# That was not a cosmetic problem. add_opponent_strength_features() joins
# team-level aggregates on the opponent NAME, so all 15 opponent-strength and
# advantage features came out NaN for 2024-25 and 2025-26 -- and the later
# dropna(subset=training_features) then deleted every row of both seasons.
# The two most recent seasons were silently absent from training entirely.
#
# Each season ships data/<season>/teams.csv with the same id -> name mapping,
# so those fill the gap.

import os

team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = data_master_team_list[data_master_team_list['season'] == season]
    mapping = dict(zip(season_team_data['team'], season_team_data['team_name']))

    if not mapping:
        teams_path = os.path.join('data', season, 'teams.csv')
        if os.path.exists(teams_path):
            teams_df = pd.read_csv(teams_path, encoding='latin-1', on_bad_lines='skip')
            mapping = dict(zip(teams_df['id'], teams_df['name']))
            print(f"  {season}: not in master_team_list, "
                  f"used {teams_path} ({len(mapping)} teams)")
        else:
            print(f"  WARNING: {season} has no team mapping and no {teams_path}")
    team_id_name_mapping[season] = mapping


# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)


all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# Nothing downstream works if an id survives here, so check rather than assume.
unmapped = (
    all_seasons_data['opponent_team']
    .astype(str)
    .str.fullmatch(r'\d+(\.\d+)?')
)
if unmapped.any():
    offending = (
        all_seasons_data.loc[unmapped]
        .groupby('season')
        .size()
        .to_dict()
    )
    raise ValueError(
        f"opponent_team is still a numeric id for {int(unmapped.sum()):,} rows: "
        f"{offending}.\n"
        f"  Opponent-strength features join on the team name, so these rows "
        f"would end up all-NaN and be dropped from training."
    )

print(f"opponent_team mapped to names for all {len(all_seasons_data):,} rows")
print(f"  seasons covered: {sorted(team_id_name_mapping)}")

# Save Current Data State

Export the processed data to CSV for checkpoint and further processing.

In [ ]:
# save all seasons data to a csv file
# utf-8, matching cell 47 and every reader below. This used to write
# latin-1, which only worked because cell 47 rewrote the file as utf-8
# before anything read it -- and the pipeline scripts skip cell 47.
all_seasons_data.to_csv('all_seasons_data.csv', index=False, encoding='utf-8')

## 2. Chronological match index

game_number is each player's match number within a season, assigned from
kickoff time. Every lag and rolling feature sorts on it, and the models
drop rows below game_number 5 because their history is too short.

In [ ]:
# Assign game_number using kickoff_time for ALL seasons

print("="*80)
print("Assigning game_number using kickoff_time approach")
print("="*80)

# Parse kickoff_time to datetime for sorting
print("\nParsing kickoff_time...")
all_seasons_updated = all_seasons_df.copy()
all_seasons_updated['kickoff_datetime'] = pd.to_datetime(all_seasons_updated['kickoff_time'])

# Sort by player, season, and kickoff_time (chronological order)
print("Sorting records by player, season, and kickoff_time...")
all_seasons_updated = all_seasons_updated.sort_values(['element', 'season', 'kickoff_datetime'])

# Assign game_number for each player-season combination
print("Assigning game_number...")
all_seasons_updated['game_number'] = (
    all_seasons_updated
    .groupby(['element', 'season'])
    .cumcount() + 1
)

# Convert to nullable integer
all_seasons_updated['game_number'] = all_seasons_updated['game_number'].astype('Int64')

# Clean up temporary column
all_seasons_updated = all_seasons_updated.drop('kickoff_datetime', axis=1)

# Verify results
print(f"\n SUCCESS: game_number assigned to all records!")
print(f"   Total records: {len(all_seasons_updated):,}")
print(f"   Records with game_number: {all_seasons_updated['game_number'].notna().sum():,}")
print(f"   Records missing game_number: {all_seasons_updated['game_number'].isna().sum():,}")

# Show distribution
print(f"\nGame number statistics:")
print(f"   Min: {all_seasons_updated['game_number'].min()}")
print(f"   Max: {all_seasons_updated['game_number'].max()}")
print(f"   Mean: {all_seasons_updated['game_number'].mean():.1f}")

# Sample verification
print("\nSample verification - Mohamed Salah 2017-18 (first 5 games):")
salah_sample = all_seasons_updated[
    (all_seasons_updated['name'].str.contains('Salah', na=False)) &
    (all_seasons_updated['season'] == '2017-18')
][['name', 'game_number', 'kickoff_time', 'opponent_team', 'GW']].head(5)
print(salah_sample.to_string(index=False))

print("\n" + "="*80)

## 3. Feature engineering

Lags 1-5 and rolling 3/5/10 of every match statistic, opponent strength,
price momentum, availability, expected goals, and rebuilt fixture
difficulty. Every feature is shifted one match inside the player group,
so a row only ever sees matches already played.

In [ ]:
# Written with encoding='latin-1' by older versions of this pipeline;
# pandas defaults to utf-8 on read, so fall back rather than crash on the
# first accented player name.
try:
    all_seasons_data = pd.read_csv('all_seasons_data_final.csv')
except UnicodeDecodeError:
    all_seasons_data = pd.read_csv('all_seasons_data_final.csv', encoding='latin-1')

In [ ]:
# Create my_team_score and opponent_team_score columns based on was_home
all_seasons_data['my_team_score'] = all_seasons_data.apply(
    lambda row: row['team_h_score'] if row['was_home'] else row['team_a_score'],
    axis=1
)
all_seasons_data['opponent_team_score'] = all_seasons_data.apply(
    lambda row: row['team_a_score'] if row['was_home'] else row['team_h_score'],
    axis=1
)

In [ ]:
# Create result indicator: 1 = win, 0 = draw, -1 = loss
all_seasons_data['result'] = all_seasons_data.apply(
    lambda row: 1 if row['my_team_score'] > row['opponent_team_score']
                else (0 if row['my_team_score'] == row['opponent_team_score'] else -1),
    axis=1
)

## Define Previous Game Statistics Function

Create lagged features from previous games for each player.

In [ ]:
def add_previous_game_stats(df, n_gameweeks=5, use_cross_season=False):

    # Columns to exclude from previous gameweek calculation
    exclude_columns = ['name', 'element', 'GW', 'game_number', 'position', 'team',
                       'season', 'fixture', 'kickoff_time', 'round', 'team_h_score', 'team_a_score']

    # Get stat columns (only numeric columns except the excluded ones)
    stat_columns = [col for col in df.columns
                    if col not in exclude_columns and pd.api.types.is_numeric_dtype(df[col])]

    # Determine groupby columns and sort order based on cross_season parameter
    if use_cross_season:
        # Use 'name' for cross-season continuity (element IDs change between seasons)
        # Sort by name, season, game_number for correct chronological order
        df_sorted = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['name']
    else:
        # Group by element and season - stats only within same season
        df_sorted = df.sort_values(['element', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['element', 'season']

    # Create a copy to avoid modifying the original
    df_result = df_sorted.copy()

    # For each stat column, create N previous game columns
    for stat in stat_columns:
        for i in range(1, n_gameweeks + 1):
            col_name = f'{stat}_prev_{i}'
            # Shift by i positions for each player (based on game_number order)
            df_result[col_name] = df_result.groupby(group_cols, sort=False)[stat].shift(i)

    return df_result

### Apply the function

In [ ]:
# Load data if needed
# all_seasons_data = pd.read_csv('all_seasons_data.csv', index_col=0)

# Apply the function to create lagged features
# Parameters:
#   n_gameweeks: Number of previous gameweeks to include (default=5)
#   use_cross_season: Whether to carry stats across seasons (default=False)
all_seasons_data_with_prev = add_previous_game_stats(all_seasons_data, n_gameweeks=5, use_cross_season=True)

# Check the result
print("Original shape:", all_seasons_data.shape)
print("New shape:", all_seasons_data_with_prev.shape)
print("\nSample of new columns:")
print(all_seasons_data_with_prev.filter(regex='_prev_').columns.tolist()[:20])

## Feature Engineering: Opponent Strength
Calculate opponent strength metrics based on team performance to capture fixture difficulty.

In [ ]:
def add_opponent_strength_features(df, rolling_windows=[3, 5]):

    # Sort in-place to save memory, then reset index
    df.sort_values(['season', 'game_number', 'team'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Step 1: Calculate team-level aggregates per game_number
    # We use game_number to ensure correct ordering even with postponed matches
    # goals_scored uses 'sum' because we sum all player goals
    # goals_conceded uses 'max' because it's team-level (same for all players)
    # clean_sheets is derived from goals_conceded (not from player data, as players
    # can have CS=1 if subbed after 60min but before team conceded)
    team_stats = df.groupby(['season', 'game_number', 'team']).agg({
        'goals_scored': 'sum',           # Sum of all player goals = team goals
        'goals_conceded': 'max',         # Team-level stat (same for all players)
        'total_points': 'sum',           # Sum of all player FPL points
    }).reset_index()

    # Rename for clarity
    team_stats.columns = ['season', 'game_number', 'team', 'team_goals_scored',
                          'team_goals_conceded', 'team_total_points']

    # Derive team clean sheet from goals_conceded (CS=1 only if goals_conceded=0)
    team_stats['team_clean_sheet'] = (team_stats['team_goals_conceded'] == 0).astype(int)

    # Step 2: Sort by team, season, and game_number for correct rolling calculation
    team_stats.sort_values(['team', 'season', 'game_number'], inplace=True)

    # Step 3: Calculate rolling averages for each window size
    # min_periods=1 means rolling calculates even with fewer values than window size:
    #   - Game 2's rolling_3 uses only Game 1 (1 value, not 3)
    #   - Game 3's rolling_3 uses Games 1-2 (2 values, not 3)
    #   - Game 4's rolling_3 uses Games 1-3 (full 3 values)
    # shift(1) pushes values forward, so we use PAST games only (Game 1 → NaN)
    team_rolling_cols = ['team_goals_scored', 'team_goals_conceded', 'team_clean_sheet', 'team_total_points']

    for window in rolling_windows:
        for col in team_rolling_cols:
            team_stats[f'{col}_rolling_{window}'] = (
                team_stats.groupby(['team', 'season'])[col]
                .transform(lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))
            )

    # Step 4: Calculate strength ratings (using rolling_3 as primary window for strength)
    primary_window = rolling_windows[0]  # Use first window (3) for strength calculations

    # Defensive strength = inverse of goals conceded (lower conceded = higher strength)
    team_stats['defensive_strength'] = (
        1 / (team_stats[f'team_goals_conceded_rolling_{primary_window}'] + 1)
    )

    # Offensive strength = goals scored rolling average
    team_stats['offensive_strength'] = (
        team_stats[f'team_goals_scored_rolling_{primary_window}']
    )

    # Overall team strength = total points rolling average
    team_stats['overall_team_strength'] = (
        team_stats[f'team_total_points_rolling_{primary_window}']
    )

    # Step 5: Prepare columns for merge (NO NaN filling - keep NaN for transparency)
    own_team_cols = ['season', 'game_number', 'team',
                     'defensive_strength', 'offensive_strength', 'overall_team_strength']
    for window in rolling_windows:
        for col in team_rolling_cols:
            own_team_cols.append(f'{col}_rolling_{window}')

    # Step 6: Create opponent stats dataframe with 'opponent_' prefix
    opponent_stats = team_stats[['season', 'game_number', 'team',
                                  'defensive_strength', 'offensive_strength', 'overall_team_strength'] +
                                 [f'{col}_rolling_{w}' for w in rolling_windows for col in team_rolling_cols]].copy()

    rename_dict = {'team': 'opponent_team'}
    for col in opponent_stats.columns:
        if col not in ['season', 'game_number', 'team']:
            rename_dict[col] = 'opponent_' + col
    opponent_stats.rename(columns=rename_dict, inplace=True)

    # Step 7: Merge own team stats
    df = df.merge(
        team_stats[own_team_cols],
        on=['season', 'game_number', 'team'],
        how='left'
    )

    # Step 8: Merge opponent stats
    df = df.merge(
        opponent_stats,
        on=['season', 'game_number', 'opponent_team'],
        how='left'
    )

    # Step 9: Calculate relative strength features (advantage)
    # These will be NaN if either team's strength is NaN
    df['offensive_advantage'] = df['offensive_strength'] - df['opponent_defensive_strength']
    df['defensive_advantage'] = df['defensive_strength'] - df['opponent_offensive_strength']
    df['overall_advantage'] = df['overall_team_strength'] - df['opponent_overall_team_strength']

    # Step 10: Add difficulty rating (normalized)
    # Higher value = more difficult opponent
    df['opponent_difficulty'] = (
        df['opponent_overall_team_strength'] / df['overall_team_strength']
    )

    # Clean up memory
    del team_stats, opponent_stats

    return df

### Apply Opponent Strength Features
Add team and opponent strength metrics to the dataset.

In [ ]:
# Free up memory before applying opponent strength features
import gc

# Delete the previous version to free memory
if 'all_seasons_data_with_opponent' in dir():
    del all_seasons_data_with_opponent
gc.collect()

# Apply opponent strength features to a copy of the data (to avoid modifying original)
all_seasons_data_with_opponent = add_opponent_strength_features(
    all_seasons_data_with_prev.copy(),
    rolling_windows=[3, 5]  # Rolling windows for team stats
)

# Check new features
print("Shape after adding opponent features:", all_seasons_data_with_opponent.shape)
print("\nNew team/opponent columns added:")
team_opponent_cols = [col for col in all_seasons_data_with_opponent.columns
                      if any(x in col.lower() for x in ['team_goals', 'team_clean', 'team_total_points',
                                                         'opponent', 'advantage', 'difficulty', 'strength'])]
for col in sorted(team_opponent_cols):
    print(f"  - {col}")

In [ ]:
# VALIDATION: Team Rolling Features & NaN Analysis

print("=" * 80)
print("VALIDATION: Team Rolling Features for Opponent Strength")
print("=" * 80)

# 1. Check a specific team's rolling stats across first 6 games
team_to_check = "Arsenal"
season_to_check = "2023-24"

arsenal_games = all_seasons_data_with_opponent[
    (all_seasons_data_with_opponent['team'] == team_to_check) &
    (all_seasons_data_with_opponent['season'] == season_to_check)
][['game_number', 'team', 'team_goals_scored_rolling_3', 'team_goals_scored_rolling_5',
   'team_goals_conceded_rolling_3', 'team_goals_conceded_rolling_5',
   'team_clean_sheet_rolling_3', 'team_clean_sheet_rolling_5']].drop_duplicates().sort_values('game_number')

print(f"\n1. {team_to_check}'s First 6 Games in {season_to_check} - Team Rolling Stats:")
print("-" * 80)
display(arsenal_games.head(6))

# 2. Show actual game results with DERIVED team clean sheet
print(f"\n2. {team_to_check}'s Actual Game Results (with DERIVED team_clean_sheet):")
print("-" * 80)
arsenal_actual = all_seasons_data_with_opponent[
    (all_seasons_data_with_opponent['team'] == team_to_check) &
    (all_seasons_data_with_opponent['season'] == season_to_check)
].groupby('game_number').agg({
    'goals_scored': 'sum',
    'goals_conceded': 'max',
}).reset_index().sort_values('game_number')

# Derive team_clean_sheet correctly: 1 only if goals_conceded == 0
arsenal_actual['team_clean_sheet'] = (arsenal_actual['goals_conceded'] == 0).astype(int)
display(arsenal_actual.head(6))

print("\nNote: team_clean_sheet is derived as 1 ONLY when goals_conceded = 0")
print("The raw 'clean_sheets' column from player data can be 1 even when team conceded")
print("(if player was subbed after 60min but before team conceded)")

# 3. Explain NaN behavior with min_periods=1
print("\n3. NaN Behavior Explanation (min_periods=1):")
print("-" * 80)
print("""
With min_periods=1 and shift(1):
- Game 1: NaN (no previous game after shift)
- Game 2: Uses Game 1 only (1 value, not full 3 for rolling_3)
- Game 3: Uses Games 1-2 (2 values, not full 3 for rolling_3)
- Game 4: Uses Games 1-3 (FULL 3 values for rolling_3)

So rolling_3 at Game 2 = Game 1's value
   rolling_3 at Game 3 = avg(Game 1, Game 2)
   rolling_3 at Game 4 = avg(Game 1, Game 2, Game 3) ← First true rolling_3
""")

# 4. Verify the calculation manually
print("4. Manual Verification of Rolling Calculation:")
print("-" * 80)
if len(arsenal_actual) >= 4:
    g1_goals = arsenal_actual[arsenal_actual['game_number'] == 1]['goals_scored'].values[0]
    g2_goals = arsenal_actual[arsenal_actual['game_number'] == 2]['goals_scored'].values[0]
    g3_goals = arsenal_actual[arsenal_actual['game_number'] == 3]['goals_scored'].values[0]

    expected_g2 = g1_goals  # Only Game 1
    expected_g3 = (g1_goals + g2_goals) / 2  # Avg of Games 1-2
    expected_g4 = (g1_goals + g2_goals + g3_goals) / 3  # Avg of Games 1-3

    actual_g2 = arsenal_games[arsenal_games['game_number'] == 2]['team_goals_scored_rolling_3'].values[0]
    actual_g3 = arsenal_games[arsenal_games['game_number'] == 3]['team_goals_scored_rolling_3'].values[0]
    actual_g4 = arsenal_games[arsenal_games['game_number'] == 4]['team_goals_scored_rolling_3'].values[0]

    print(f"Game 2 rolling_3: Expected={expected_g2:.2f}, Actual={actual_g2:.2f} {'✓' if abs(expected_g2-actual_g2)<0.01 else '✗'}")
    print(f"Game 3 rolling_3: Expected={expected_g3:.2f}, Actual={actual_g3:.2f} {'✓' if abs(expected_g3-actual_g3)<0.01 else '✗'}")
    print(f"Game 4 rolling_3: Expected={expected_g4:.2f}, Actual={actual_g4:.2f} {'✓' if abs(expected_g4-actual_g4)<0.01 else '✗'}")

# 5. Verify clean sheet rolling calculation
print("\n5. Clean Sheet Rolling Verification:")
print("-" * 80)
g1_cs = arsenal_actual[arsenal_actual['game_number'] == 1]['team_clean_sheet'].values[0]
g2_cs = arsenal_actual[arsenal_actual['game_number'] == 2]['team_clean_sheet'].values[0]
g3_cs = arsenal_actual[arsenal_actual['game_number'] == 3]['team_clean_sheet'].values[0]

expected_cs_g2 = g1_cs  # Only Game 1
expected_cs_g3 = (g1_cs + g2_cs) / 2  # Avg of Games 1-2
expected_cs_g4 = (g1_cs + g2_cs + g3_cs) / 3  # Avg of Games 1-3

actual_cs_g2 = arsenal_games[arsenal_games['game_number'] == 2]['team_clean_sheet_rolling_3'].values[0]
actual_cs_g3 = arsenal_games[arsenal_games['game_number'] == 3]['team_clean_sheet_rolling_3'].values[0]
actual_cs_g4 = arsenal_games[arsenal_games['game_number'] == 4]['team_clean_sheet_rolling_3'].values[0]

print(f"Game 1: goals_conceded={arsenal_actual[arsenal_actual['game_number']==1]['goals_conceded'].values[0]} → team_clean_sheet={g1_cs}")
print(f"Game 2: goals_conceded={arsenal_actual[arsenal_actual['game_number']==2]['goals_conceded'].values[0]} → team_clean_sheet={g2_cs}")
print(f"Game 3: goals_conceded={arsenal_actual[arsenal_actual['game_number']==3]['goals_conceded'].values[0]} → team_clean_sheet={g3_cs}")
print()
print(f"Game 2 CS rolling_3: Expected={expected_cs_g2:.2f}, Actual={actual_cs_g2:.2f} {'✓' if abs(expected_cs_g2-actual_cs_g2)<0.01 else '✗'}")
print(f"Game 3 CS rolling_3: Expected={expected_cs_g3:.2f}, Actual={actual_cs_g3:.2f} {'✓' if abs(expected_cs_g3-actual_cs_g3)<0.01 else '✗'}")
print(f"Game 4 CS rolling_3: Expected={expected_cs_g4:.2f}, Actual={actual_cs_g4:.2f} {'✓' if abs(expected_cs_g4-actual_cs_g4)<0.01 else '✗'}")

## Feature Engineering: Rolling Averages (Player Form)
Calculate moving averages to capture short-term and long-term player performance trends.

In [ ]:
def add_rolling_player_stats(df, windows=[3, 5, 10]):

    # Key performance metrics to calculate rolling averages for
    rolling_stats = ['total_points', 'goals_scored', 'assists', 'minutes',
                     'bonus', 'bps', 'clean_sheets', 'saves',
                     'ict_index', 'creativity', 'threat', 'influence']

    # Sort by player NAME (consistent across seasons), then season and game_number
    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)

    # Calculate rolling averages for each window size
    for window in windows:
        for stat in rolling_stats:
            if stat in df.columns:
                col_name = f'{stat}_rolling_{window}'
                # Calculate rolling mean using transform() with shift INSIDE the group
                # This ensures shift only happens within each player's data
                df[col_name] = (
                    df.groupby('name')[stat]
                    .transform(lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))
                )

    return df

## Feature Engineering: Additional Context Features
Add home/away indicators, season progression, and price momentum features.

In [ ]:
def add_context_features(df):

    # 1. Home/Away indicator
    df['is_home'] = df['was_home'].astype(int)

    # 2. Season progression features
    # Season stage: early (GW 1-13), mid (GW 14-26), late (GW 27-38)
    df['season_stage'] = pd.cut(df['GW'], bins=[0, 13, 26, 38],
                                 labels=['early', 'mid', 'late'])

    # Gameweek as percentage of season completion
    df['season_progress'] = df['GW'] / 38.0

    # 3. Price change features (if value column exists)
    if 'value' in df.columns:
        # Sort by player and time
        df = df.sort_values(['element', 'season', 'GW']).reset_index(drop=True)

        # Price change from previous gameweek
        df['price_change'] = df.groupby(['element', 'season'])['value'].diff()

        # Cumulative price change within season (from starting price)
        df['price_change_cumulative'] = df.groupby(['element', 'season'])['value'].transform(
            lambda x: x - x.iloc[0] if len(x) > 0 else 0
        )

        # Price trend: increasing (1), stable (0), decreasing (-1)
        df['price_trend'] = df['price_change'].apply(
            lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
        )
    return df

## Feature Engineering: Availability

Whether a player takes the field at all is the single largest driver of FPL points, and the model had no feature for it beyond raw lagged minutes. These rebuild the signal -- nailed on, rotated, or frozen out -- from appearance history, all shifted one match so nothing sees the present.

In [ ]:
def add_availability_features(df):
    """Will this player be on the pitch at all?

    FPL's own xP beats these models by 0.03-0.15 R2, and roughly half of that
    edge is simply knowing the starting XI: in 2024-25, players who did not
    play carried a mean xP of 0.145 against 2.299 for players who did, and xP's
    R2 falls from 0.42 to 0.20 once you look only at players who appeared.

    Minutes are the single biggest driver of FPL points, and the model had no
    feature describing whether a player is nailed on, rotated, or frozen out --
    only raw lagged minutes. These reconstruct that from appearance history.

    Every feature is shifted by one match inside the player group, so a row
    only ever sees matches that had already been played. `minutes` is used
    rather than `starts` because starts does not exist before 2023-24, and
    restricting to it would throw away seven seasons.
    """
    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)

    appeared = (df['minutes'] > 0).astype(float)
    started = (df['minutes'] >= 60).astype(float)
    grouped = df.groupby('name')

    def prior_mean(series, window):
        """Mean over the previous `window` matches, excluding this one."""
        return series.groupby(df['name']).transform(
            lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
        )

    # How often they have been on the pitch lately.
    for window in (3, 5, 10):
        df[f'avail_played_rate_{window}'] = prior_mean(appeared, window)
        df[f'avail_started_rate_{window}'] = prior_mean(started, window)

    # Erratic minutes are the rotation signal; a nailed starter has low variance.
    df['avail_minutes_std_5'] = grouped['minutes'].transform(
        lambda x: x.rolling(window=5, min_periods=2).std().shift(1)
    )

    # Recent minutes against the longer baseline: positive means working back
    # into the side, negative means dropping out of it.
    df['avail_minutes_trend'] = (
        grouped['minutes'].transform(
            lambda x: x.rolling(window=3, min_periods=1).mean().shift(1))
        - grouped['minutes'].transform(
            lambda x: x.rolling(window=10, min_periods=1).mean().shift(1))
    )

    def run_length(flag):
        """Length of the current unbroken run of `flag`, up to the last match."""
        def _run(x):
            blocks = (~x.astype(bool)).cumsum()
            return x.groupby(blocks).cumsum().shift(1)
        return flag.groupby(df['name']).transform(_run)

    # Consecutive blanks is the closest thing here to an injury flag.
    df['avail_zero_streak'] = run_length(1.0 - appeared)
    df['avail_start_streak'] = run_length(started)

    # Matches since they last completed an hour.
    since = grouped['minutes'].transform(
        lambda x: x.ge(60).astype(int).shift(1).fillna(0)
    )
    df['avail_games_since_start'] = since.groupby(df['name']).transform(
        lambda x: x.groupby(x.cumsum()).cumcount()
    )

    # Share of this season's matches they have featured in so far. Reset per
    # season, because last season's role says little after a transfer.
    df['avail_season_played_rate'] = appeared.groupby(
        [df['name'], df['season']]
    ).transform(lambda x: x.expanding().mean().shift(1))

    # Three coarse roles, which the linear models can use directly.
    rate5 = df['avail_started_rate_5']
    played5 = df['avail_played_rate_5']
    df['avail_is_nailed'] = (rate5 >= 0.8).astype(int)
    df['avail_is_rotation_risk'] = ((played5 >= 0.2) & (played5 < 0.8)).astype(int)
    df['avail_is_frozen_out'] = (played5 < 0.2).astype(int)

    # Days of rest. Long gaps follow injuries; short ones mean congestion.
    if 'kickoff_time' in df.columns:
        kickoff = pd.to_datetime(df['kickoff_time'], errors='coerce', utc=True)
        df['avail_days_since_last'] = (
            kickoff - kickoff.groupby(df['name']).shift(1)
        ).dt.total_seconds() / 86400.0

    created = [c for c in df.columns if c.startswith('avail_')]
    print(f"Added {len(created)} availability features:")
    for c in created:
        filled = df[c].notna().mean() * 100
        print(f"  {c:<32} {filled:5.1f}% populated")
    return df


print("Availability feature function defined!")

## Feature Engineering: Expected Goals

Conditional on a player appearing, the model explained 5% of variance. Expected goals and assists measure the chances created rather than whether they went in, which is the signal a scoreline discards.

In [ ]:
def add_expected_features(df):
    """Lagged and rolling xG / xA.

    Diagnostics put the model's R2 among players who actually appeared at
    0.052: conditional on playing, it barely beat the mean. Expected goals and
    assists target exactly that gap. They measure the chances a player got
    rather than whether they went in, so they carry the signal a scoreline
    throws away -- a striker with 0.8 xG and no goal is a better bet next week
    than one who scored from his only touch.

    Named xg_* so the feature selector picks the family up by prefix, and
    shifted a match inside the player group like every other feature here.
    """
    source = [c for c in ('expected_goals', 'expected_assists',
                          'expected_goal_involvements', 'expected_goals_conceded')
              if c in df.columns]
    if not source:
        print("No expected-goals columns present; skipping.")
        return df

    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
    grouped = df.groupby('name')

    for col in source:
        short = col.replace('expected_', 'x').replace('goal_involvements', 'gi') \
                   .replace('goals_conceded', 'gc').replace('goals', 'g') \
                   .replace('assists', 'a')
        values = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

        for lag in (1, 2, 3):
            df[f'xg_{short}_prev_{lag}'] = values.groupby(df['name']).shift(lag)
        for window in (3, 5, 10):
            df[f'xg_{short}_rolling_{window}'] = values.groupby(df['name']).transform(
                lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))

    # Overperformance: goals scored against the chances taken. A large positive
    # run is usually finishing luck rather than skill, and tends to come back.
    if {'goals_scored', 'expected_goals'} <= set(df.columns):
        goals = grouped['goals_scored'].transform(
            lambda x: x.rolling(window=10, min_periods=1).sum().shift(1))
        xg = grouped['expected_goals'].transform(
            lambda x: x.rolling(window=10, min_periods=1).sum().shift(1))
        df['xg_overperformance_10'] = goals - xg

    # Whether this row's season predates the stat at all. Without it a model
    # reads the zeros of 2016-17 as "took no shots" rather than "not recorded".
    if 'has_xg' in df.columns:
        df['xg_is_recorded'] = df['has_xg'].astype(int)

    created = [c for c in df.columns if c.startswith('xg_')]
    print(f"Added {len(created)} expected-goals features:")
    for c in created:
        nonzero = (df[c].fillna(0) != 0).mean() * 100
        print(f"  {c:<30} {nonzero:5.1f}% non-zero")
    return df


print("Expected-goals feature function defined!")

## Feature Engineering: Fixture Difficulty (rebuilt)

The previous opponent features scored -0.009 R2 on their own. These replace them with venue-split rolling attack and defence form, and the gap between a side's attack and the opposition's defence.

In [ ]:
def add_fixture_features(df):
    """Rebuilt opponent strength, because the old version measured nothing.

    Ablation put the previous opponent_* family at -0.009 R2 alone and +0.001
    marginally: seventeen features carrying no signal. FPL's own xP, which uses
    fixture difficulty properly, reaches 0.20 conditional R2 against this
    model's 0.052, so difficulty is not the problem -- the encoding was.

    Three things the old features got wrong:

      1. They mixed a team's whole-season record with its recent form, so a
         side that started badly and improved looked average all year.
      2. They ignored home and away, which is most of the effect being
         claimed -- conceding at home and conceding away are different rates.
      3. They were built from the opponent's aggregate points rather than
         from what a player in this position actually needs to know: how many
         goals that opponent concedes, and how often they keep a clean sheet.

    Everything below is a rolling mean over matches already played, shifted by
    one, computed per team and split by venue.
    """
    df = df.sort_values(['team', 'season', 'game_number']).reset_index(drop=True)

    # One row per team-match: what that team did in that fixture.
    # my_team_score / opponent_team_score are set in the cell that flips
    # team_h_score and team_a_score by venue, so they are the goals for and
    # against in this fixture regardless of which side the player was on.
    needed = {'my_team_score', 'opponent_team_score'}
    if not needed <= set(df.columns):
        print(f"missing {sorted(needed - set(df.columns))}; fixture features skipped")
        return df

    team_match = (df.groupby(['season', 'team', 'game_number', 'was_home'])
                    .agg(scored=('my_team_score', 'max'),
                         conceded=('opponent_team_score', 'max'))
                    .reset_index())
    if team_match['scored'].isna().all():
        print("team goal columns are empty; fixture features skipped")
        return df

    team_match = team_match.sort_values(['season', 'team', 'game_number'])
    grp = team_match.groupby(['season', 'team'])

    for window in (4, 8):
        team_match[f'att_form_{window}'] = grp['scored'].transform(
            lambda x: x.rolling(window, min_periods=1).mean().shift(1))
        team_match[f'def_form_{window}'] = grp['conceded'].transform(
            lambda x: x.rolling(window, min_periods=1).mean().shift(1))
        team_match[f'cs_rate_{window}'] = grp['conceded'].transform(
            lambda x: x.eq(0).rolling(window, min_periods=1).mean().shift(1))

    # Same three, split by venue: a team's away defence is its own statistic.
    venue = team_match.groupby(['season', 'team', 'was_home'])
    team_match['att_form_venue'] = venue['scored'].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    team_match['def_form_venue'] = venue['conceded'].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1))

    form_cols = [c for c in team_match.columns
                 if c.startswith(('att_form', 'def_form', 'cs_rate'))]

    # Attach the player's own team form...
    own = team_match[['season', 'team', 'game_number', 'was_home'] + form_cols]
    own = own.rename(columns={c: f'fx_own_{c}' for c in form_cols})
    df = df.merge(own, on=['season', 'team', 'game_number', 'was_home'], how='left')

    # ...and the opponent's, taken from the opponent's own row in that match,
    # which is the same game_number with the venue flipped.
    opp = team_match[['season', 'team', 'game_number', 'was_home'] + form_cols].copy()
    opp['was_home'] = ~opp['was_home'].astype(bool)
    opp = opp.rename(columns={'team': 'opponent_team',
                              **{c: f'fx_opp_{c}' for c in form_cols}})
    df = df.merge(opp, on=['season', 'opponent_team', 'game_number', 'was_home'],
                  how='left')

    # What a player actually wants: the gap between his side's attack and the
    # opposition's defence, and vice versa.
    if {'fx_own_att_form_8', 'fx_opp_def_form_8'} <= set(df.columns):
        df['fx_attack_edge'] = df['fx_own_att_form_8'] - df['fx_opp_def_form_8']
        df['fx_defence_edge'] = df['fx_opp_att_form_8'] - df['fx_own_def_form_8']
        df['fx_cs_chance'] = df['fx_own_cs_rate_8'] - df['fx_opp_att_form_8']

    df['fx_is_home'] = df['was_home'].astype(int)

    created = [c for c in df.columns if c.startswith('fx_')]
    print(f"Added {len(created)} fixture features:")
    for c in created:
        print(f"  {c:<32} {df[c].notna().mean() * 100:5.1f}% populated")
    return df


print("Fixture feature function defined!")

### Apply All Feature Engineering Functions
Combine all feature engineering steps to create the complete dataset.

In [ ]:
# Apply rolling averages for player form
print("Adding rolling averages...")
all_seasons_data_with_rolling = add_rolling_player_stats(
    all_seasons_data_with_opponent,
    windows=[3, 5, 10]
)

# Apply context features
print("Adding context features...")
all_seasons_data_with_context = add_context_features(all_seasons_data_with_rolling)

# Availability: how likely this player is to be on the pitch at all.
print("Adding availability features...")
all_seasons_data_with_avail = add_availability_features(all_seasons_data_with_context)

print("Adding expected-goals features...")
all_seasons_data_with_xg = add_expected_features(all_seasons_data_with_avail)

print("Adding rebuilt fixture features...")
all_seasons_data_featured = add_fixture_features(all_seasons_data_with_xg)

# Check final shape
print(f"\nFinal dataset shape: {all_seasons_data_featured.shape}")
print(f"Original dataset shape: {all_seasons_data.shape}")
print(f"Total new features added: {all_seasons_data_featured.shape[1] - all_seasons_data.shape[1]}")

In [ ]:
# save the final dataset with features
all_seasons_data_featured.to_csv('all_seasons_data_featured.csv', index=False)

## 4. Split, preprocess and train

Whole-season holdout, scalers and searches fitted on the training fold
only, TimeSeriesSplit for the inner CV. Per-position models, because a
goalkeeper and a forward score points in entirely different ways.

In [ ]:
import pandas as pd
# Written with encoding='latin-1' by older versions of this pipeline;
# pandas defaults to utf-8 on read, so fall back rather than crash on the
# first accented player name.
try:
    all_seasons_data_featured = pd.read_csv('all_seasons_data_featured.csv')
except UnicodeDecodeError:
    all_seasons_data_featured = pd.read_csv('all_seasons_data_featured.csv',
                                            encoding='latin-1')

In [ ]:
all_seasons_data_featured.columns.tolist()
print(" all columns in the final dataset:")
print(all_seasons_data_featured.columns.tolist())
# display  the non numeric columns only
non_numeric_cols = [col for col in all_seasons_data_featured.columns if not pd.api.types.is_numeric_dtype(all_seasons_data_featured[col])]
print("Non-numeric columns in the final dataset:")
print(non_numeric_cols)

In [ ]:
# Get column names
print("Total columns:", len(all_seasons_data_featured.columns))
print(all_seasons_data_featured.columns.tolist())

# Get non-numeric columns
non_numeric_cols = [col for col in all_seasons_data_featured.columns if not pd.api.types.is_numeric_dtype(all_seasons_data_featured[col])]
print("\n\nNon-numeric columns:", non_numeric_cols)

## Data Preprocessing for Regression Models
We'll prepare all features for predicting player total_points, ensuring we only use data from previous gameweeks.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Create a copy to work with
df = all_seasons_data_featured.copy()

print(f"Initial dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")

### Step 1: Identify Features to Exclude from Training
These are the features that represent current or future gameweek data (not available at prediction time)

In [ ]:
# Columns to exclude from features (these are current gameweek stats or identifiers)
# These features contain information about the CURRENT gameweek, not previous ones
exclude_from_training = [
    'total_points',  # Target variable
    'name',  # Identifier
    'element',  # Player ID
    'fixture',  # Match ID
    'kickoff_time',  # Time info
    'round',  # Round number
    'GW',  # Gameweek number
    'game_number',  # Game number
    'match_number',  # Match number

    # Current gameweek performance stats (not available at prediction time)
    'assists',  # Current GW assists
    'bonus',  # Current GW bonus
    'bps',  # Current GW BPS
    'clean_sheets',  # Current GW clean sheets
    'clearances_blocks_interceptions',  # Current GW defensive stats
    'creativity',  # Current GW creativity
    'goals_conceded',  # Current GW goals conceded
    'goals_scored',  # Current GW goals scored
    'ict_index',  # Current GW ICT
    'influence',  # Current GW influence
    'minutes',  # Current GW minutes played
    'own_goals',  # Current GW own goals
    'penalties_missed',  # Current GW penalties missed
    'penalties_saved',  # Current GW penalties saved
    'recoveries',  # Current GW recoveries
    'red_cards',  # Current GW red cards
    'saves',  # Current GW saves
    'selected',  # Current GW selection
    'tackles',  # Current GW tackles
    'team_a_score',  # Current GW away score
    'team_h_score',  # Current GW home score
    'threat',  # Current GW threat
    'transfers_balance',  # Current GW transfers
    'transfers_in',  # Current GW transfers in
    'transfers_out',  # Current GW transfers out
    'yellow_cards',  # Current GW yellow cards
    'defensive_contribution',  # Current GW defensive contribution
    'result',
    'my_team_score',
    'opponent_team_score'
]

# Verify all these columns exist
exclude_from_training = [col for col in exclude_from_training if col in df.columns]

print(f"Excluding {len(exclude_from_training)} columns that represent current gameweek data:")
print(exclude_from_training)

### Step 2: Handle Non-Numerical Features
Convert all categorical/text features to numerical representations

In [ ]:
# Get non-numeric columns (excluding those already in exclude list)
non_numeric_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
print("Non-numeric columns found:")
print(non_numeric_cols)

# Create encoders dictionary to store all label encoders
label_encoders = {}

# Encode categorical features
for col in non_numeric_cols:
    if col not in exclude_from_training:
        print(f"\nEncoding {col}...")
        le = LabelEncoder()
        # Handle NaN values
        df[col] = df[col].fillna('missing')
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"  - Created {col}_encoded with {len(le.classes_)} unique values")

# For was_home (boolean), convert to int if not already
if 'was_home' in df.columns and df['was_home'].dtype == 'bool':
    df['was_home'] = df['was_home'].astype(int)

# For is_home (boolean), convert to int if not already
if 'is_home' in df.columns and df['is_home'].dtype == 'bool':
    df['is_home'] = df['is_home'].astype(int)

# Handle was_home_prev columns (convert True/False strings to 0/1)
was_home_prev_cols = [col for col in df.columns if 'was_home_prev_' in col]
for col in was_home_prev_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0})
        df[col] = df[col].fillna(0).astype(float)

print(f"\nEncoding complete. Created {len(label_encoders)} encoded features.")

### Step 3: Select Training Features
Select only features that use previous gameweek data

In [ ]:
# Get all potential feature columns (numeric only)
# This includes all columns except those explicitly excluded
all_possible_features = [col for col in df.columns if col not in exclude_from_training]

# Keep only numeric columns (includes our newly encoded features)
training_features = []
for col in all_possible_features:
    if pd.api.types.is_numeric_dtype(df[col]):
        training_features.append(col)
    elif col not in non_numeric_cols:  # Skip original non-numeric columns (we have encoded versions)
        training_features.append(col)

# Remove the original non-numeric columns from training features (keep encoded versions)
original_non_numeric = [col for col in non_numeric_cols if col not in exclude_from_training]
training_features = [col for col in training_features if col not in original_non_numeric]

print(f"Total training features: {len(training_features)}")
print(f"\nFeature categories:")

# Categorize features for better understanding
prev_features = [col for col in training_features if '_prev_' in col]
rolling_features = [col for col in training_features if '_rolling_' in col]
opponent_features = [col for col in training_features if 'opponent_' in col]
encoded_features = [col for col in training_features if '_encoded' in col]
strength_features = [col for col in training_features if 'strength' in col or 'advantage' in col]
other_features = [col for col in training_features if col not in prev_features + rolling_features +
                  opponent_features + encoded_features + strength_features]

print(f"  - Previous gameweek features (_prev_): {len(prev_features)}")
print(f"  - Rolling average features (_rolling_): {len(rolling_features)}")
print(f"  - Opponent features: {len(opponent_features)}")
print(f"  - Strength/advantage features: {len(strength_features)}")
print(f"  - Encoded categorical features: {len(encoded_features)}")
print(f"  - Other features: {len(other_features)}")

print(f"\nOther features include: {other_features[:20]}")  # Show first 20

### Step 4: Handle Missing Values
Fill missing values with appropriate strategies for each feature type

In [ ]:
# Check missing values in training features
print("Missing values in training features:")
missing_counts = df[training_features].isnull().sum()
features_with_missing = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(f"\nFeatures with missing values: {len(features_with_missing)}")
if len(features_with_missing) > 0:
    print("\nTop 20 features with most missing values:")
    print(features_with_missing.head(20))

# Drop rows with game_number < 5
print(f"\nOriginal dataset shape: {df.shape}")
if 'game_number' in df.columns:
    rows_before = len(df)
    df = df[df['game_number'] >= 5]
    rows_dropped = rows_before - len(df)
    print(f"Dropped {rows_dropped} rows with game_number < 5")
    print(f"Dataset shape after filtering game_number >= 5: {df.shape}")
else:
    print("Warning: 'game_number' column not found in dataframe")

# Drop rows with any NaN/null values in training features or target
print("\nDropping rows with missing values...")
rows_before = len(df)
df = df.dropna(subset=training_features + ['total_points'])
rows_dropped = rows_before - len(df)
print(f"Dropped {rows_dropped} rows with missing values")

print(f"\nAfter cleaning:")
print(f"  Dataset shape: {df.shape}")
print(f"  Missing values in features: {df[training_features].isnull().sum().sum()}")
print(f"  Missing values in target: {df['total_points'].isnull().sum()}")

### Step 5: Prepare Features and Target
Create X (features) and y (target) datasets

In [ ]:
import numpy as np
# Prepare feature matrix X and target vector y
X = df[training_features].copy()
y = df['total_points'].copy()

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nTarget variable statistics:")
print(y.describe())

# Check for any remaining infinite values
print(f"\nInfinite values in X: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

# Replace any infinite values with NaN then fill with median
if np.isinf(X.select_dtypes(include=[np.number])).sum().sum() > 0:
    print("Replacing infinite values with median...")
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].median())

print(f"\nFinal feature check:")
print(f"  X shape: {X.shape}")
print(f"  X missing values: {X.isnull().sum().sum()}")
print(f"  X infinite values: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

In [ ]:
# here I will make a final verification of the data
print(f"Final dataset shape: {df.shape}")
# print the

### Step 6: Standardize Features
Apply StandardScaler to normalize all features to the same range

## Temporal Splitting

Every split below is by whole season, never shuffled. See the next cell for why:
a random split on player-gameweek rows leaks neighbouring gameweeks across the
train/test boundary, because the features are lags and rolling averages of the
same recent matches.

| fold | seasons |
| --- | --- |
| train | 2016-17 .. 2022-23 |
| validation | 2023-24 |
| test | 2024-25, 2025-26 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Temporal splitting helpers
# ─────────────────────────────────────────────────────────────────────────────
# This is panel time-series data: one row per player per gameweek. A random
# train_test_split puts GW12 of a season into train and GW13 of the SAME season
# into test. Those two rows share almost all of their information, because the
# features are lags and rolling averages of the same recent matches -- so the
# model is scored on weeks it has effectively already seen. Every R2 produced
# that way is optimistic.
#
# Splitting on whole seasons instead answers the question we actually care
# about: given everything up to now, how well do we predict a season we have
# never seen?

SEASON_ORDER = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25', '2025-26',
]

# Two split regimes. The default uses every season available.
#
# "xg" exists because expected goals only start in 2022-23. Training on the
# full history with xG present means 84% of training rows carry a fabricated
# zero and the test season carries real values -- a distribution shift, and
# measurably harmful: xG scores 0.242 R2 on its own but -0.0015 marginally
# under the default split. Confining train, validation and test to the seasons
# that actually record it costs a lot of rows and removes the mismatch.
#
# Set FPL_SPLIT=xg to use it.
import os as _os_split

_SPLITS = {
    'full': (['2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
              '2021-22', '2022-23'], ['2023-24'], ['2024-25', '2025-26']),
    'xg':   (['2022-23', '2023-24'], ['2024-25'], ['2025-26']),
}
SPLIT_NAME = _os_split.environ.get('FPL_SPLIT', 'full').lower()
if SPLIT_NAME not in _SPLITS:
    raise ValueError(f"FPL_SPLIT must be one of {sorted(_SPLITS)}, got {SPLIT_NAME!r}")
TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS = _SPLITS[SPLIT_NAME]
print(f"split: {SPLIT_NAME.upper()}  train={TRAIN_SEASONS} val={VAL_SEASONS} test={TEST_SEASONS}")

# Number of folds for the inner cross-validation used during hyperparameter
# search. TimeSeriesSplit, not KFold: the inner CV has to respect time order
# too, otherwise the leak just moves from the outer split to the inner one.
INNER_CV_SPLITS = 3


def temporal_masks(frame):
    """Boolean train/val/test masks for `frame`, split on whole seasons.

    Masks are aligned to the frame's own index, so they can be applied to
    anything sharing that index (X, y, ...).
    """
    if 'season' not in frame.columns:
        raise ValueError(
            "temporal_masks() needs a 'season' column. If you are passing a "
            "feature matrix, pass the row frame it came from instead."
        )

    season = frame['season'].astype(str)

    unknown = sorted(set(season.unique()) - set(SEASON_ORDER))
    if unknown:
        raise ValueError(f"unrecognised season(s) {unknown}; update SEASON_ORDER")

    return season.isin(TRAIN_SEASONS), season.isin(VAL_SEASONS), season.isin(TEST_SEASONS)


def chronological_order(frame):
    """Index of `frame` sorted by season then gameweek.

    Reordering rows into time order is what makes TimeSeriesSplit meaningful
    for the inner CV: fold k must be entirely earlier than fold k+1.
    """
    rank = {s: i for i, s in enumerate(SEASON_ORDER)}
    key = pd.DataFrame({
        '_season': frame['season'].astype(str).map(rank),
        '_gw': frame['GW'] if 'GW' in frame.columns else 0,
    }, index=frame.index)
    return key.sort_values(['_season', '_gw']).index


def describe_split(train_mask, val_mask, test_mask, label=''):
    """Print the size of each fold, and warn about rows that fell through."""
    n_train, n_val, n_test = int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())
    total = len(train_mask)
    print(f"{label}Temporal split ({total} rows):")
    for name, n, seasons in (
        ('train', n_train, TRAIN_SEASONS),
        ('val', n_val, VAL_SEASONS),
        ('test', n_test, TEST_SEASONS),
    ):
        pct = 100 * n / total if total else 0.0
        span = seasons[0] if len(seasons) == 1 else f"{seasons[0]}..{seasons[-1]}"
        print(f"  {name:<5} {n:>7,} rows ({pct:4.1f}%)  {span}")

    leftover = total - n_train - n_val - n_test
    if leftover:
        print(f"  WARNING: {leftover:,} rows fell outside every fold")
    if min(n_train, n_val, n_test) == 0:
        raise ValueError(f"{label}a fold is empty -- check the season values in this frame")


print("Temporal split helpers defined.")
print(f"  train: {TRAIN_SEASONS[0]}..{TRAIN_SEASONS[-1]}  ({len(TRAIN_SEASONS)} seasons)")
print(f"  val:   {VAL_SEASONS}")
print(f"  test:  {TEST_SEASONS}")
print(f"  inner CV: TimeSeriesSplit(n_splits={INNER_CV_SPLITS})")

In [ ]:
# Standardise using statistics from the TRAINING seasons only.
#
# The previous version called scaler.fit_transform(X) on the whole dataset and
# only split afterwards, so the mean and variance of the test seasons were
# baked into every training row. A scaler must never see data it will later be
# evaluated on.

train_mask, val_mask, test_mask = temporal_masks(df.loc[X.index])
describe_split(train_mask, val_mask, test_mask)

scaler = StandardScaler()
scaler.fit(X.loc[train_mask])

X_scaled = pd.DataFrame(
    scaler.transform(X),
    columns=X.columns,
    index=X.index,
)

print("\nFeature standardization complete (fitted on training seasons only).")
print("\nTraining-fold statistics after scaling (should be ~0 mean, ~1 std):")
print(X_scaled.loc[train_mask].describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print("\nTest-fold statistics after scaling (drift from 0/1 here is real, not a bug):")
print(X_scaled.loc[test_mask].describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print(f"\nScaled features shape: {X_scaled.shape}")

### Step 7: Train-Test Split
Split data into training and testing sets (80-20 split)

In [ ]:
# Apply the temporal split computed in the previous cell.
#
# Was: train_test_split(X_scaled, y, test_size=0.2, shuffle=True). On panel
# time-series data that leaks neighbouring gameweeks across the boundary; see
# the temporal split helper cell for the full reasoning.

X_train, y_train = X_scaled.loc[train_mask], y.loc[train_mask]
X_val, y_val = X_scaled.loc[val_mask], y.loc[val_mask]
X_test, y_test = X_scaled.loc[test_mask], y.loc[test_mask]

print("Data split complete (by season, no shuffling).\n")
for name, Xs, ys, seasons in (
    ('Training', X_train, y_train, TRAIN_SEASONS),
    ('Validation', X_val, y_val, VAL_SEASONS),
    ('Testing', X_test, y_test, TEST_SEASONS),
):
    print(f"{name} set  ({', '.join(seasons)}):")
    print(f"  X shape: {Xs.shape}")
    print(f"  y mean:  {ys.mean():.3f}   y std: {ys.std():.3f}")
    print(f"  share:   {100 * len(Xs) / len(X_scaled):.1f}% of {len(X_scaled)} rows\n")

# The target mean drifting between folds is expected and worth seeing: FPL
# scoring rules changed over these seasons, so a model trained on 2016-22 is
# predicting a slightly different game than 2024-26.

### Summary of Data Preprocessing
Review what we've done

In [ ]:
print("="*70)
print("DATA PREPROCESSING SUMMARY")
print("="*70)

print("\n1. FEATURES USED:")
print(f"   - Total features: {len(training_features)}")
print(f"   - Previous gameweek stats (_prev_): {len(prev_features)}")
print(f"   - Rolling averages (_rolling_): {len(rolling_features)}")
print(f"   - Opponent features: {len(opponent_features)}")
print(f"   - Strength/advantage features: {len(strength_features)}")
print(f"   - Encoded categorical features: {len(encoded_features)}")
print(f"   - Other predictive features: {len(other_features)}")

print("\n2. FEATURES EXCLUDED (current gameweek data):")
print(f"   - {len(exclude_from_training)} features excluded")
print(f"   - These represent CURRENT gameweek performance")
print(f"   - Not available at prediction time")

print("\n3. DATA TRANSFORMATIONS:")
print(f"    Non-numeric features encoded to numeric")
print(f"    Missing values handled appropriately")
print(f"    All features standardized (mean=0, std=1)")
print(f"    No infinite values")
print(f"    Boolean features converted to 0/1")

print("\n4. DATASET SPLIT:")
print(f"   - Total samples: {len(X_scaled):,}")
print(f"   - Training: {len(X_train):,} samples ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"   - Testing: {len(X_test):,} samples ({len(X_test)/len(X_scaled)*100:.1f}%)")

print("\n5. TARGET VARIABLE (total_points):")
print(f"   - Mean: {y.mean():.3f}")
print(f"   - Std: {y.std():.3f}")
print(f"   - Min: {y.min():.0f}")
print(f"   - Max: {y.max():.0f}")

print("\n6. KEY PRINCIPLE:")
print("    All training features use ONLY previous gameweek data")
print("    No data leakage from current/future gameweeks")
print("    Model will predict future performance based on past")

print("\n" + "="*70)
print("READY FOR MODEL TRAINING!")
print("="*70)

## Regression Model Training
Train multiple regression models to predict player total_points

In [ ]:
# Import regression models and evaluation metrics
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time


In [ ]:
from collections import defaultdict

def auto_group_features(columns):
    groups = defaultdict(list)

    for col in columns:
        base = col.split("_prev")[0]
        base = base.split("_rolling")[0]
        groups[base].append(col)

    return dict(groups)
# Group features based on common prefixes
feature_groups = auto_group_features(training_features)
print("Feature groups identified:")
for group, cols in feature_groups.items():
    print(f"  - {group}: {len(cols)} features")
# print the keys as list
print("Feature group keys:", list(feature_groups.keys()))


In [ ]:
# Availability features are named avail_* and have no _prev/_rolling
# suffix, so auto_group_features gives each its own single-member group.
# Collect them by prefix rather than listing eighteen .get() calls.
AVAILABILITY_FEATURES = sorted(c for c in training_features
                               if c.startswith('avail_'))
# Same prefix trick for the two new families.
EXPECTED_FEATURES = sorted(c for c in training_features
                          if c.startswith('xg_'))
FIXTURE_FEATURES = sorted(c for c in training_features
                         if c.startswith('fx_'))
print(f"Availability features picked up: {len(AVAILABILITY_FEATURES)}")
print(f"Expected-goals features picked up: {len(EXPECTED_FEATURES)}")
print(f"Fixture features picked up: {len(FIXTURE_FEATURES)}")


COMMON_FEATURES  = feature_groups.get('value', []) + \
    feature_groups.get('bonus', []) + \
    feature_groups.get('bps', []) + \
    feature_groups.get('clearances_blocks_interceptions', []) + \
    feature_groups.get('minutes', []) + \
    feature_groups.get('own_goals', []) + \
    feature_groups.get('red_cards', []) + \
    feature_groups.get('selected', []) + \
    feature_groups.get('total_points', []) + \
    feature_groups.get('transfers_balance', []) + \
    feature_groups.get('transfers_in', []) + \
    feature_groups.get('transfers_out', []) + \
    feature_groups.get('yellow_cards', []) + \
    feature_groups.get('overall_team_strength', []) + \
    feature_groups.get('team_total_points', []) + \
    feature_groups.get('opponent_overall_team_strength', []) + \
    feature_groups.get('opponent_team_total_points', []) + \
    feature_groups.get('offensive_advantage', []) + \
    feature_groups.get('defensive_advantage', []) + \
    feature_groups.get('overall_advantage', []) + \
    feature_groups.get('opponent_difficulty', []) + \
    feature_groups.get('is_home', []) + \
    feature_groups.get('price_change', []) + \
    feature_groups.get('price_change_cumulative', []) + \
    feature_groups.get('price_trend', []) + \
    feature_groups.get('my_team_score', []) + \
    feature_groups.get('opponent_team_score', []) + \
    feature_groups.get('result', []) + \
    AVAILABILITY_FEATURES + \
    EXPECTED_FEATURES + \
    FIXTURE_FEATURES




ONLY_NOT_GK_FEATURES = feature_groups.get('assists', []) + \
    feature_groups.get('creativity', []) + \
    feature_groups.get('goals_scored', []) + \
    feature_groups.get('ict_index', []) + \
    feature_groups.get('influence', []) + \
    feature_groups.get('penalties_missed', []) + \
    feature_groups.get('recoveries', []) + \
    feature_groups.get('tackles', []) + \
    feature_groups.get('threat', []) + \
    feature_groups.get('defensive_contribution', []) + \
    feature_groups.get('offensive_strength', []) + \
    feature_groups.get('team_goals_scored', []) + \
    feature_groups.get('opponent_defensive_strength', []) + \
    feature_groups.get('opponent_team_goals_conceded', [])



ONLY_NOT_FWD_FEATURES = feature_groups.get('clean_sheets', []) + \
    feature_groups.get('goals_conceded', []) + \
    feature_groups.get('defensive_strength', []) + \
    feature_groups.get('team_goals_conceded', []) + \
    feature_groups.get('team_clean_sheet', []) + \
    feature_groups.get('opponent_offensive_strength', []) + \
    feature_groups.get('opponent_team_goals_scored', []) + \
    feature_groups.get('opponent_team_clean_sheet', [])


ONLY_GPK_FEATURES = feature_groups.get('saves', []) + \
    feature_groups.get('penalties_saved', []) + \
    feature_groups.get('team_saves', [])

GK_FEATURES = COMMON_FEATURES + ONLY_GPK_FEATURES + ONLY_NOT_FWD_FEATURES
DEF_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
MID_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
FWD_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES



POSITION_FEATURES = {
    'GK': GK_FEATURES,
    'DEF': DEF_FEATURES,
    'MID': MID_FEATURES,
    'FWD': FWD_FEATURES
}

# ---------------------------------------------------------------------------
# Feature-set size
#
# scripts/ablate.py measured every family two ways: what it adds on top of
# everything else, and what it scores on its own. No family is worth more than
# +0.005 marginally, while minutes history alone reaches 0.326 against the full
# model's 0.339 -- the 220 columns are largely restatements of each other.
#
# Keeping only the families that carry signal costs about 0.003 R2 and drops
# 180 columns. The families left out include the 17 opponent-strength and
# fixture-difficulty features, which score below the mean on their own, and the
# FBref defensive columns, worth -0.0001 marginally.
#
# The FBref *data* still matters and is not going anywhere: defensive
# contribution decides a +2 point bonus on 10,604 rows, so it is part of the
# target even though it is not worth much as a feature.
#
# Set FPL_FEATURE_SET=full in the environment to train on all 220 again.
# ---------------------------------------------------------------------------
import os as _os

COMPACT_PREFIXES = (
    'minutes', 'avail_', 'total_points_', 'bps', 'value', 'ict_index',
    # Added after measurement: the rebuilt fixture features are worth
    # +0.0074 R2 marginally, the largest of any group, and 37x what the
    # opponent_* family they replaced managed (+0.0002).
    'fx_',
)

FEATURE_SET = _os.environ.get('FPL_FEATURE_SET', 'compact').lower()

if FEATURE_SET == 'compact':
    _full_counts = {pos: len(feats) for pos, feats in POSITION_FEATURES.items()}
    POSITION_FEATURES = {
        pos: [f for f in feats if f.startswith(COMPACT_PREFIXES)]
        for pos, feats in POSITION_FEATURES.items()
    }
    print(f"feature set: COMPACT {COMPACT_PREFIXES}")
    for pos, feats in POSITION_FEATURES.items():
        print(f"  {pos}: {_full_counts[pos]} -> {len(feats)} features")
    for pos, feats in POSITION_FEATURES.items():
        assert len(feats) >= 20, (
            f"{pos} kept only {len(feats)} features; the compact prefixes match "
            f"almost nothing, which means the feature names have changed"
        )
else:
    print(f"feature set: FULL ({FEATURE_SET!r})")

print("Position-specific feature sets defined!")
for pos, features in POSITION_FEATURES.items():
    print(f"{pos}: {len(features)} features")

## Prepare Position-Specific Training Data

Create train/validation/test splits for each player position.

In [ ]:
# Prepare data for modeling

# If a position's feature list is largely absent from the frame we were handed,
# that is a bug in the caller, not something to work around. Silently training
# on whatever happened to survive is how the "direct" models ended up fitted on
# a single column ('value') while reporting themselves as 201-feature models.
MIN_FEATURE_COVERAGE = 0.90


def prepare_position_data(df, position, features, min_coverage=MIN_FEATURE_COVERAGE):
    """Prepare X, y for a single position.

    Raises if fewer than `min_coverage` of the requested features exist in
    `df`, which almost always means the wrong dataframe was passed in.
    """

    # Filter by position
    pos_df = df[df['position'] == position].copy()

    # Get available features (some may not exist)
    available_features = [f for f in features if f in pos_df.columns]
    missing_features = [f for f in features if f not in pos_df.columns]

    coverage = len(available_features) / len(features) if features else 0.0
    if coverage < min_coverage:
        preview = ', '.join(missing_features[:8])
        more = f" (+{len(missing_features) - 8} more)" if len(missing_features) > 8 else ""
        raise ValueError(
            f"{position}: only {len(available_features)}/{len(features)} requested "
            f"features exist in this dataframe ({coverage:.1%} < {min_coverage:.0%}).\n"
            f"  Missing: {preview}{more}\n"
            f"  This usually means an un-engineered dataframe was passed. The "
            f"position feature lists are built from the FEATURED frame ('df'), "
            f"not from 'all_seasons_data'."
        )

    if missing_features:
        print(f"  note: {len(missing_features)} of {len(features)} features absent, "
              f"training on {len(available_features)}")

    # Remove rows with missing target
    pos_df = pos_df.dropna(subset=['total_points'])

    # Fill missing features with 0
    for col in available_features:
        pos_df[col] = pos_df[col].fillna(0)

    # Remove infinite values
    pos_df = pos_df.replace([np.inf, -np.inf], 0)

    X = pos_df[available_features]
    y = pos_df['total_points']

    return X, y, available_features, pos_df


print("Data preparation function defined!")
print(f"  guard: raises if <{MIN_FEATURE_COVERAGE:.0%} of requested features are present")

## Hyperparameter Grids Definition

Define parameter grids for hyperparameter tuning with GridSearchCV.

In [ ]:
# Define hyperparameter grids for different models
PARAM_GRIDS = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 9],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100]
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1],
        'l1_ratio': [0.2, 0.5, 0.8]
    }
}

PARAM_GRIDS['XGBoost'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

PARAM_GRIDS['LightGBM'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9, 1.0]
}

print("Hyperparameter grids defined for models:")
for model_name in PARAM_GRIDS:
    print(f"  - {model_name}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
def tune_model(model, param_grid, X_train, y_train, cv=5):
    """Perform GridSearchCV for hyperparameter tuning"""

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def tune_model_randomized(model, param_distributions, X_train, y_train, n_iter=50, cv=5):
    """Perform RandomizedSearchCV for faster hyperparameter tuning"""

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

print("Hyperparameter tuning functions defined!")

## Import Additional Model Libraries

Import gradient boosting and additional model libraries.

In [ ]:
# Additional imports for model training
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
import xgboost as xgb
import lightgbm as lgb

print("Additional libraries imported successfully!")

## Train/Validation/Test Split and Model Training

Split data and train multiple regression models for each position.

In [ ]:
# Define the models to train
MODELS = {
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet(max_iter=10000),
    #'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    #'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

# Reduced parameter grids for faster training (use full grids for production)
#
# Every hyperparameter in the previous grids came back sitting on an edge:
# Ridge chose alpha=10 (the maximum), and ElasticNet and XGBoost chose the
# minimum of every one of their ranges, in all eight fits. When the search
# always stops at the boundary, the grid is in the wrong place -- the optimum
# is somewhere outside it. Every range below is extended in the direction the
# search was pulling, which for the tree models means more regularisation.
#
# LightGBM previously had no grid at all (this entry was commented out), so it
# trained at defaults with unlimited leaf growth. It memorised the training
# seasons: mean train R2 0.577 against test 0.312, a gap of 0.26, where the
# tuned models sat at 0.00-0.04. num_leaves and min_child_samples are the two
# knobs that actually bound that.
PARAM_GRIDS_REDUCED = {
    'Ridge': {'alpha': [1, 10, 100, 1000]},
    'ElasticNet': {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.1, 0.3, 0.5]},
    #'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20], 'min_samples_split': [2, 5]},
    #'GradientBoosting': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    'XGBoost': {'n_estimators': [200, 400], 'learning_rate': [0.02, 0.05], 'max_depth': [2, 3, 4]},
    'LightGBM': {'n_estimators': [200, 400], 'learning_rate': [0.02, 0.05], 'num_leaves': [15, 31],
                 'min_child_samples': [50]}
}

print("Models and reduced parameter grids defined!")
print(f"Models to train: {list(MODELS.keys())}")
for _name, _grid in PARAM_GRIDS_REDUCED.items():
    _combos = 1
    for _v in _grid.values():
        _combos *= len(_v)
    print(f"  {_name:<11} {_combos:>3} combinations x {INNER_CV_SPLITS} folds "
          f"= {_combos * INNER_CV_SPLITS} fits")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit


def train_and_evaluate_models(df, positions, position_features, models, param_grids, use_tuning=False):

    results = {}

    for position in positions:
        print(f"\n{'='*80}")
        print(f"Training models for position: {position}")
        print(f"{'='*80}")

        # Prepare data for this position
        features = position_features[position]
        X, y, available_features, pos_df = prepare_position_data(df, position, features)

        print(f"Data shape: X={X.shape}, y={y.shape}")
        print(f"Available features: {len(available_features)}")

        if len(X) < 100:
            print(f"Skipping {position} - not enough data")
            continue

        # ── Temporal split ────────────────────────────────────────────────
        # Whole seasons, never shuffled. Rows are also put into time order so
        # the inner TimeSeriesSplit sees genuinely earlier -> later folds.
        order = chronological_order(pos_df)
        X, y = X.loc[order], y.loc[order]
        train_mask, val_mask, test_mask = (m.loc[order] for m in temporal_masks(pos_df))
        describe_split(train_mask, val_mask, test_mask, label=f"{position}: ")

        X_train, y_train = X[train_mask], y[train_mask]
        X_val, y_val = X[val_mask], y[val_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        # Scale features -- fitted on the training fold only
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        position_results = {
            'scaler': scaler,
            'features': available_features,
            'X_test': X_test,
            'y_test': y_test,
            'split': {
                'train_seasons': TRAIN_SEASONS,
                'val_seasons': VAL_SEASONS,
                'test_seasons': TEST_SEASONS,
                'n_train': int(train_mask.sum()),
                'n_val': int(val_mask.sum()),
                'n_test': int(test_mask.sum()),
            },
            'models': {}
        }

        # Train each model
        for model_name, model in models.items():
            print(f"\n--- Training {model_name} ---")
            start_time = time.time()

            try:
                # Clone the model to avoid issues with refitting
                model_clone = type(model)(**model.get_params())

                if use_tuning and model_name in param_grids:
                    # Perform hyperparameter tuning.
                    # TimeSeriesSplit, not KFold: the training fold is in time
                    # order, and a shuffled inner CV would reintroduce exactly
                    # the leak the outer split just removed.
                    print(f"Tuning hyperparameters...")
                    grid_search = GridSearchCV(
                        model_clone, param_grids[model_name],
                        cv=TimeSeriesSplit(n_splits=INNER_CV_SPLITS),
                        scoring='neg_mean_squared_error', n_jobs=-1
                    )
                    grid_search.fit(X_train_scaled, y_train)
                    best_model = grid_search.best_estimator_
                    best_params = grid_search.best_params_
                    print(f"Best params: {best_params}")
                else:
                    # Train without tuning
                    best_model = model_clone
                    best_model.fit(X_train_scaled, y_train)
                    best_params = None

                # Predictions
                y_train_pred = best_model.predict(X_train_scaled)
                y_val_pred = best_model.predict(X_val_scaled)
                y_test_pred = best_model.predict(X_test_scaled)

                # Calculate metrics
                train_metrics = {
                    'r2': r2_score(y_train, y_train_pred),
                    'mae': mean_absolute_error(y_train, y_train_pred),
                    'mse': mean_squared_error(y_train, y_train_pred),
                    'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred))
                }

                val_metrics = {
                    'r2': r2_score(y_val, y_val_pred),
                    'mae': mean_absolute_error(y_val, y_val_pred),
                    'mse': mean_squared_error(y_val, y_val_pred),
                    'rmse': np.sqrt(mean_squared_error(y_val, y_val_pred))
                }

                test_metrics = {
                    'r2': r2_score(y_test, y_test_pred),
                    'mae': mean_absolute_error(y_test, y_test_pred),
                    'mse': mean_squared_error(y_test, y_test_pred),
                    'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
                }

                training_time = time.time() - start_time

                position_results['models'][model_name] = {
                    'model': best_model,
                    'best_params': best_params,
                    'train_metrics': train_metrics,
                    'val_metrics': val_metrics,
                    'test_metrics': test_metrics,
                    'training_time': training_time,
                    'y_test_pred': y_test_pred
                }

                print(f"Training R²: {train_metrics['r2']:.4f}, Validation R²: {val_metrics['r2']:.4f}, Test R²: {test_metrics['r2']:.4f}")
                print(f"Test MAE: {test_metrics['mae']:.4f}, Test RMSE: {test_metrics['rmse']:.4f}")
                print(f"Training time: {training_time:.2f}s")

            except Exception as e:
                print(f"Error training {model_name}: {str(e)}")
                continue

        results[position] = position_results

    return results

print("Training and evaluation function defined!")
print("  split: by season (temporal)   inner CV: TimeSeriesSplit")

## Direct Model Training Results

Review the training results for direct point prediction models.

In [ ]:
# Train models for all positions
# Set use_tuning=True for hyperparameter tuning (slower but potentially better results)
# Set use_tuning=False for faster training with default parameters

positions_to_train = ['GK', 'DEF', 'MID', 'FWD']

print("Starting model training for all positions...")
print("This may take a few minutes...\n")

# NOTE: `df` -- the FEATURED, encoded frame built earlier in this notebook.
#
# This used to pass `all_seasons_data`, which is the raw merged frame from
# BEFORE feature engineering. None of the lag/rolling/opponent columns in
# POSITION_FEATURES exist there, so prepare_position_data silently dropped all
# but one of them and every "direct" model was really a single-feature model
# fitted on 'value' (price). prepare_position_data now raises instead.
all_results = train_and_evaluate_models(
    df=df,
    positions=positions_to_train,
    position_features=POSITION_FEATURES,
    models=MODELS,
    param_grids=PARAM_GRIDS_REDUCED,
    use_tuning=True  # Set to True for hyperparameter tuning
)

print("\n" + "="*80)
print("Model training completed for all positions!")
print("="*80)

## Model Comparison Summary

Compare model performance across positions and select best models.

In [ ]:
def create_results_summary(results):
    """Create a summary DataFrame of all model results"""

    summary_data = []

    for position, pos_results in results.items():
        for model_name, model_results in pos_results['models'].items():
            summary_data.append({
                'Position': position,
                'Model': model_name,
                'Train_R2': model_results['train_metrics']['r2'],
                'Val_R2': model_results['val_metrics']['r2'],
                'Test_R2': model_results['test_metrics']['r2'],
                'Train_MAE': model_results['train_metrics']['mae'],
                'Val_MAE': model_results['val_metrics']['mae'],
                'Test_MAE': model_results['test_metrics']['mae'],
                'Train_RMSE': model_results['train_metrics']['rmse'],
                'Val_RMSE': model_results['val_metrics']['rmse'],
                'Test_RMSE': model_results['test_metrics']['rmse'],
                'Training_Time': model_results['training_time']
            })

    summary_df = pd.DataFrame(summary_data)
    return summary_df

# Create and display summary
summary_df = create_results_summary(all_results)
print("="*100)
print("MODEL PERFORMANCE SUMMARY - ALL POSITIONS")
print("="*100)
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('model_results_summary.csv', index=False)
print("\nResults saved to 'model_results_summary.csv'")

In [ ]:
# Display best model for each position
print("\n" + "="*80)
print("BEST MODEL FOR EACH POSITION (Based on Test R²)")
print("="*80)

for position in positions_to_train:
    if position in all_results:
        pos_models = all_results[position]['models']
        if pos_models:
            best_model_name = max(pos_models.keys(), key=lambda x: pos_models[x]['test_metrics']['r2'])
            best_r2 = pos_models[best_model_name]['test_metrics']['r2']
            best_mae = pos_models[best_model_name]['test_metrics']['mae']
            best_rmse = pos_models[best_model_name]['test_metrics']['rmse']
            print(f"\n{position}:")
            print(f"  Best Model: {best_model_name}")
            print(f"  Test R²: {best_r2:.4f}")
            print(f"  Test MAE: {best_mae:.4f}")
            print(f"  Test RMSE: {best_rmse:.4f}")

## 5. Two-stage models

A single regressor over every row spends its capacity on the easy half of
the problem: 60% of rows are players who did not appear. Measured, that
gave R2 0.35 overall but 0.076 among players who actually took the field.

Splitting it into P(plays) and E[points | plays] lifts the conditional
figure to 0.083 and improves every position. The play classifier reaches
0.954 AUC on its own, which is the part of the problem this project was
always solving well.

The haul classifier reaches 0.869 AUC but does **not** improve captain
ranking over expected points -- both catch about a third of hauls in the
top 5%. It is kept because the probability is more honest than an expected
value for a decision about upside, not because it scores better.

In [ ]:
class HurdleModel:
    """P(plays) x E[points | plays], fitted separately.

    Why bother. A single regressor over every row spends its capacity on the
    easy half of the problem: 60% of rows are players who did not appear and
    score almost exactly zero. Diagnostics showed the consequence -- overall R2
    0.333, but only 0.052 among players who actually took the field, and a
    predicted spread just 0.55x the real one. The model had learned to say
    "probably about one point" very accurately and could not pick a captain.

    Splitting it lets each half be trained on the question it is actually
    answering:

        stage 1   classifier   will this player be on the pitch at all?
        stage 2   regressor    given that he is, how many points?

    Stage 2 trains only on rows where the player appeared, so the near-zeros
    stop drowning the signal. The product is still an expected value, and it is
    still calibrated, but the two parts can now be inspected and improved
    separately -- and stage 1's probability is useful on its own.
    """

    def __init__(self, classifier=None, regressor=None, threshold_minutes=1):
        self.classifier = classifier
        self.regressor = regressor
        self.threshold_minutes = threshold_minutes
        self.scaler_c = StandardScaler()
        self.scaler_r = StandardScaler()

    def fit(self, X, y_points, minutes):
        played = (minutes >= self.threshold_minutes).astype(int)

        self.classifier.fit(self.scaler_c.fit_transform(X), played)

        mask = played.astype(bool)
        if mask.sum() < 50:
            raise ValueError(f"only {mask.sum()} rows with minutes; cannot fit stage 2")
        self.regressor.fit(self.scaler_r.fit_transform(X[mask]), y_points[mask])

        self.play_rate_ = float(played.mean())
        self.n_stage2_ = int(mask.sum())
        return self

    def predict_parts(self, X):
        p_play = self.classifier.predict_proba(self.scaler_c.transform(X))[:, 1]
        points_if_playing = self.regressor.predict(self.scaler_r.transform(X))
        # A negative score is possible in FPL but never the sensible forecast
        # for a player who starts; clipping keeps the product interpretable.
        return p_play, np.clip(points_if_playing, 0, None)

    def predict(self, X):
        p_play, conditional = self.predict_parts(X)
        return p_play * conditional


class HaulModel:
    """P(scoring 10 or more), which is what a captain pick actually needs.

    Expected points and upside come apart badly here. The regressor's ceiling
    prediction is around 8 while real returns reach 25, and its mean forecast
    for players who did haul was 3.17 against 1.17 for everyone else -- enough
    to rank, nowhere near enough to distinguish a captain from a steady four.

    Ranking by P(haul) instead asks the question directly. It is a rare event
    (1.7% of rows), so the probability is small everywhere; what matters is the
    ordering, and the lift over the base rate.
    """

    def __init__(self, classifier=None, haul_points=10):
        self.classifier = classifier
        self.haul_points = haul_points
        self.scaler = StandardScaler()

    def fit(self, X, y_points):
        hauled = (y_points >= self.haul_points).astype(int)
        if hauled.sum() < 30:
            raise ValueError(f"only {hauled.sum()} hauls in the training fold")
        self.classifier.fit(self.scaler.fit_transform(X), hauled)
        self.base_rate_ = float(hauled.mean())
        return self

    def predict_proba(self, X):
        return self.classifier.predict_proba(self.scaler.transform(X))[:, 1]


print("HurdleModel and HaulModel defined!")

## 6. Save models and artifacts

Everything `scripts/predict_gameweek.py` needs to score an unplayed
gameweek: the per-position model, its scaler, and the exact feature
list it was trained on.

In [ ]:
import joblib
import json
import os

SAVE_DIR = 'saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 80)
print("SAVING ALL MODELS & ARTIFACTS")
print("=" * 80)

# ──────────────────────────────────────────────
# 1. Save DIRECT per-position models (all_results)
# ──────────────────────────────────────────────
print("\n1. Saving direct per-position models from all_results ...")
direct_meta = {}

for position, pos_results in all_results.items():
    pos_dir = os.path.join(SAVE_DIR, 'direct', position)
    os.makedirs(pos_dir, exist_ok=True)

    # Save scaler
    joblib.dump(pos_results['scaler'], os.path.join(pos_dir, 'scaler.joblib'))

    # Save feature list
    with open(os.path.join(pos_dir, 'features.json'), 'w') as f:
        json.dump(pos_results['features'], f)

    # Save each trained model
    best_model_name = None
    best_r2 = -999
    for model_name, model_data in pos_results['models'].items():
        joblib.dump(model_data['model'], os.path.join(pos_dir, f'{model_name}.joblib'))
        if model_data['test_metrics']['r2'] > best_r2:
            best_r2 = model_data['test_metrics']['r2']
            best_model_name = model_name

    direct_meta[position] = {
        'best_model': best_model_name,
        'best_test_r2': float(best_r2),
        'features_count': len(pos_results['features']),
        'models_saved': list(pos_results['models'].keys()),
    }
    print(f"   {position}: best={best_model_name} (R2={best_r2:.4f}), saved {len(pos_results['models'])} models")

with open(os.path.join(SAVE_DIR, 'direct', 'meta.json'), 'w') as f:
    json.dump(direct_meta, f, indent=2)

# ──────────────────────────────────────────────
# 2. Save PCA per-position models (pca_model_results)
# ──────────────────────────────────────────────
if 'pca_model_results' not in globals():
    print("\n2. skipped: the PCA branch was not trained in this run")
    pca_model_results = {}

print("\n2. Saving PCA per-position models from pca_model_results ...")
pca_meta = {}

for position, pos_results in pca_model_results.items():
    pos_dir = os.path.join(SAVE_DIR, 'pca_models', position)
    os.makedirs(pos_dir, exist_ok=True)

    best_model_name = None
    best_r2 = -999
    for model_name, model_data in pos_results['models'].items():
        joblib.dump(model_data['model'], os.path.join(pos_dir, f'{model_name}.joblib'))
        if model_data['test_metrics']['r2'] > best_r2:
            best_r2 = model_data['test_metrics']['r2']
            best_model_name = model_name

    pca_meta[position] = {
        'best_model': best_model_name,
        'best_test_r2': float(best_r2),
        'n_pca_components': int(pos_results['n_pca_components']),
        'models_saved': list(pos_results['models'].keys()),
    }
    print(f"   {position}: best={best_model_name} (R2={best_r2:.4f}), PCA dims={pos_results['n_pca_components']}")

if pca_meta:
    os.makedirs(os.path.join(SAVE_DIR, 'pca_models'), exist_ok=True)
    with open(os.path.join(SAVE_DIR, 'pca_models', 'meta.json'), 'w') as f:
        json.dump(pca_meta, f, indent=2)

# ──────────────────────────────────────────────
# 3. Save PCA artifacts (position_feature_selection)
# ──────────────────────────────────────────────
if 'position_feature_selection' not in globals():
    print("\n3. skipped: no PCA artifacts were produced in this run")
    position_feature_selection = {}

print("\n3. Saving PCA transformers & scalers from position_feature_selection ...")

for position, pfs in position_feature_selection.items():
    pos_dir = os.path.join(SAVE_DIR, 'pca_artifacts', position)
    os.makedirs(pos_dir, exist_ok=True)

    joblib.dump(pfs['pca_model'], os.path.join(pos_dir, 'pca_model.joblib'))
    joblib.dump(pfs['scaler'], os.path.join(pos_dir, 'pca_scaler.joblib'))

    with open(os.path.join(pos_dir, 'pca_config.json'), 'w') as f:
        json.dump({
            'original_features': pfs['original_features'],
            'features_after_correlation': pfs['features_after_correlation'],
            'removed_by_correlation': pfs['removed_by_correlation'],
            'pca_n_components': int(pfs['pca_n_components']),
            'pca_explained_variance': pfs['pca_explained_variance'].tolist(),
        }, f, indent=2)
    print(f"   {position}: PCA {len(pfs['original_features'])} -> {pfs['pca_n_components']} dims")

# ──────────────────────────────────────────────
# 4. Save Stat-Based Predictor
# ──────────────────────────────────────────────
print("\n4. Saving ImprovedStatBasedPredictor (stat_predictor) ...")
# The stat predictor is trained further down the notebook, so it is absent
# when only the position models have been run (e.g. scripts/train.py).
if 'stat_predictor' in globals():
    joblib.dump(stat_predictor, os.path.join(SAVE_DIR, 'stat_predictor.joblib'))
    print(f"   Saved stat_predictor with {len(stat_predictor.stat_models)} stat categories")
else:
    print("   skipped: stat_predictor was not trained in this run")

# ──────────────────────────────────────────────
# 5. Save configuration / feature definitions
# ──────────────────────────────────────────────
print("\n5. Saving configuration & feature definitions ...")

# STATS_TO_PREDICT / ALL_STATS_TO_PREDICT belong to the stat-predictor
# section further down, which has not run when only the position models
# were trained. Record the temporal split too: anyone reloading these
# models needs to know which seasons they have already seen.
config = {
    'training_features': training_features,
    'POSITION_FEATURES': POSITION_FEATURES,
    'STATS_TO_PREDICT': globals().get('STATS_TO_PREDICT'),
    'ALL_STATS_TO_PREDICT': globals().get('ALL_STATS_TO_PREDICT'),
    'exclude_from_training': exclude_from_training,
    'split': {
        'train_seasons': TRAIN_SEASONS,
        'val_seasons': VAL_SEASONS,
        'test_seasons': TEST_SEASONS,
    },
}
joblib.dump(config, os.path.join(SAVE_DIR, 'config.joblib'))
print(f"   training_features: {len(training_features)} features")
print(f"   POSITION_FEATURES: {', '.join(f'{k}={len(v)}' for k, v in POSITION_FEATURES.items())}")

# ──────────────────────────────────────────────
# 6. Save Label Encoders
# ──────────────────────────────────────────────
print("\n6. Saving label encoders ...")
joblib.dump(label_encoders, os.path.join(SAVE_DIR, 'label_encoders.joblib'))
print(f"   Saved {len(label_encoders)} label encoders")

# ──────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────
print("\n" + "=" * 80)
print("ALL ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 80)
print(f"\nSave directory: {os.path.abspath(SAVE_DIR)}")
for root, dirs, files in os.walk(SAVE_DIR):
    level = root.replace(SAVE_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files:
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{subindent}{file} ({size_kb:.1f} KB)")